<a href="https://colab.research.google.com/github/tishasetia13/inside-out-emotion-classifier/blob/main/inside_out.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###GETTING OUR DATASET READY

In [ ]:
import pandas as pd

url = "https://huggingface.co/datasets/gsri-18/ISEAR-dataset-complete/resolve/main/ISEAR_dataset_complete.csv"
df = pd.read_csv(url)

In [ ]:
#df.head()
#df.shape
#df["emotion"].value_counts()
#df.isnull().sum()

In [ ]:
df = df[df['emotion'].isin(['joy', 'sadness', 'anger', 'fear', 'disgust'])]
df = df.drop(columns=['Unnamed: 2'])

In [ ]:
#df.shape
#df['emotion'].value_counts()

In [ ]:
df['content'] = df['content'].str.replace('\\n', ' ', regex=False)
df['content'] = df['content'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
#pd.set_option('display.max_colwidth', None)
#df.sample(20, random_state=42)['content']
#df['content'].str.len().describe()

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['emotion'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['emotion'], random_state=42)

print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
from transformers import AutoTokenizer

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

sample = 'Hi, my name is Tisha'
encoded = tokenizer(sample)

print(encoded)
print(tokenizer.convert_ids_to_tokens(encoded['input_ids']))

In [ ]:
import pandas as pd

lengths = [len(tokenizer.encode(text)) for text in train_df['content']]

print(pd.Series(lengths).describe())
print("95th percentile:", pd.Series(lengths).quantile(0.95))
print("99th percentile:", pd.Series(lengths).quantile(0.99))

In [ ]:
batch_texts = train_df['content'].tolist()[:4]

batch = tokenizer(
    batch_texts,
    padding='max_length',
    truncation=True,
    max_length=64,
    return_tensors='pt',
)

print(batch["input_ids"].shape)
print(batch["attention_mask"].sum(dim=1))
print(batch["input_ids"][0])

In [ ]:
labels = sorted(train_df['emotion'].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)
print(id2label)

In [ ]:
for df in [train_df, val_df, test_df]:
  df['label'] = df['emotion'].map(label2id)

print(train_df[["emotion", "label"]].head())
print(train_df["label"].value_counts().sort_index())
print("Missing labels:", train_df["label"].isna().sum(), val_df["label"].isna().sum(), test_df["label"].isna().sum())

In [ ]:
import torch

def encode_split(df):
  enc = tokenizer(
      df['content'].tolist(),
      padding='max_length',
      truncation=True,
      max_length=64,
      return_tensors='pt',
  )
  enc['labels'] = torch.tensor(df['label'].tolist())
  return enc

train_enc = encode_split(train_df)
val_enc = encode_split(val_df)
test_enc = encode_split(test_df)

for name, enc in [("train", train_enc), ("val", val_enc), ("test", test_enc)]:
    print(name, enc["input_ids"].shape, enc["attention_mask"].shape, enc["labels"].shape)

print(train_enc["labels"][:5])

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(enc, batch_size, shuffle):
  dataset = TensorDataset(enc['input_ids'], enc['attention_mask'], enc['labels'])
  return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(train_enc, batch_size=16, shuffle=True)
val_loader = make_loader(val_enc, batch_size=32, shuffle=False)
test_loader = make_loader(test_enc, batch_size=32, shuffle=False)

print("Batches per epoch (train):", len(train_loader))

ids, mask, y = next(iter(train_loader))
print(ids.shape, mask.shape, y.shape)
print(y)